# 🐉 Forja de Contos — treino no Colab

Treina **dois adaptadores LoRA** (português e inglês) e avalia base × ajustado.
Tempo total na GPU T4: cerca de 35–45 minutos.

**Antes de começar:** *Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU.*
Depois rode as células em ordem.

## 1. Conferir a GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução -> Alterar tipo -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Instalar dependências

In [ ]:
!pip install -q transformers peft datasets accelerate
# o Colab traz um torchao antigo que o peft novo recusa; o projeto não usa torchao
!pip uninstall -y -q torchao

## 3. Enviar o projeto
Rode a célula e selecione o arquivo **forja-de-contos-colab.zip** (o zip que acompanha este notebook).

In [ ]:
from google.colab import files
up = files.upload()
name = next(iter(up))
!unzip -o -q "{name}" -d /content/forja
%cd /content/forja
!ls
# começa do zero: apaga adaptadores/resultados de execuções anteriores neste mesmo ambiente
!rm -rf adapter results outputs

## 4. Gerar o dataset (≈9.600 exemplos: 4.200 PT + 4.200 EN + 1.200 poemas em PT; 7 modos)

In [ ]:
!python data/build_dataset.py --scale 2 --poems data/poems/portuguese-poems.csv

## 5. Treinar o adaptador em português
Base: `pierreguillou/gpt2-small-portuguese`. 7 épocas, ~13 min na T4 (estimativa; o PT agora tem os poemas). Ao final aparece a perplexidade de validação.
Se a célula for interrompida, **rode a mesma célula de novo**: o treino retoma do último ponto salvo.

In [ ]:
!python train.py --lang pt --epochs 7

## 6. Treinar o adaptador em inglês
Base: `gpt2`. Também retoma se for interrompido.

In [ ]:
!python train.py --lang en --epochs 7

## 6b. Backup dos adaptadores (antes de avaliar)
Se o Colab reiniciar, o treino se perde. Baixe os adaptadores agora.

In [ ]:
!zip -qr adapters_backup.zip adapter
from google.colab import files
files.download("adapters_backup.zip")

## 7. Avaliar (base × ajustado, por idioma)
Gera `results/eval.md` (perplexidade, **% de formato correto** com 24 amostras por pedido, diversidade e varredura de
parâmetros) e `results/samples.md` (textos lado a lado, ótimo para o vídeo). Pode levar uns 10 minutos.

In [ ]:
!python evaluate.py --sweep --n-format 24

In [ ]:
print(open("results/eval.md", encoding="utf-8").read())

In [ ]:
print(open("results/samples.md", encoding="utf-8").read()[:6000])

## 8. Baixar o resultado
Gera `resultado.zip` com as pastas `adapter/` (pt e en) e `results/`.
Extraia dentro da pasta do projeto no seu computador.

In [ ]:
!zip -r -q resultado.zip adapter results
from google.colab import files
files.download("resultado.zip")